# 05a — Fluid Bed Coating Digital Twin — Demo

Full process chain: **Pre-heating → Spraying → Drying → Dissolution prediction**.

Use the **r_spray** and **r_dry** sliders to explore how coating loss rates affect weight gain and dissolution.
The dotted line shows the theoretical maximum WG assuming **no coating loss**.

* **[SP] r_spray** [×10⁻⁶ kg/s] — constant attrition during spraying (order-0). Range 0–40×10⁻⁶.
* **[DR] r_dry** [×10⁻³ kg/s] — attrition per unit WG during drying (order-1). Range 0–10×10⁻³.

Slide **Sample time** to withdraw a virtual sample at any process point and see the predicted dissolution profile.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from fluid_bed.parameters          import ProcessParameters
from fluid_bed.models.preheating   import run_preheating
from fluid_bed.models.spraying     import run_spraying
from fluid_bed.models.drying_stage import run_drying

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

In [ ]:
# Hardware / physical constants — not user-adjustable
_RHO_AIR  = 1.10    # kg/m3
_CP_AIR   = 1010.0  # J/kg/K
_RHO_PART = 1050.0  # kg/m3
_CP_PART  = 1400.0  # J/kg/K
_D_BED    = 0.60    # m
print('Physical constants loaded.')

In [ ]:
# ── Dissolution model constants ───────────────────────────────────────────────
_DISS = dict(Volume_disso=1000.0, rho_EC=0.4, Permeability=1.5e-7,
             Mass_sample=1.058, Total_min=240)

def _dissolution_k(EC_mass, ssa_cm2g):
    if EC_mass <= 0:
        return np.inf
    S = _DISS['Mass_sample'] * ssa_cm2g
    return S * _DISS['Permeability'] * _DISS['rho_EC'] * ssa_cm2g / (_DISS['Volume_disso'] * EC_mass)

def _dissolution_curve(EC_mass, ssa_cm2g):
    k = _dissolution_k(EC_mass, ssa_cm2g)
    t_s = np.arange(1, _DISS['Total_min'] + 1) * 60.0
    F = 100.0 * (1.0 - np.exp(-k * t_s)) if not np.isinf(k) else np.full_like(t_s, 100.0)
    return t_s / 60.0, F, k

_sim_cache = {}

# ── Sliders ───────────────────────────────────────────────────────────────────
S  = dict(continuous_update=False)
S2 = dict(continuous_update=False, readout_format='.2f')
_DW = {'description_width': '170px'}

_w = {
    # Particle / batch
    'd_mm':          widgets.FloatSlider(1.00, min=0.80, max=1.50, step=0.05, description='Diameter (mm)',      style=_DW, **S),
    'ssa_cm2g':      widgets.FloatSlider(65.0, min=55.0, max=75.0, step=1.0,  description='SSA (cm2/g)',        style=_DW, **S),
    'T0_C':          widgets.FloatSlider(20.0, min=15.0, max=30.0, step=1.0,  description='T0 particle (C)',    style=_DW, **S),
    'batch_kg':      widgets.FloatSlider(4.5,  min=3.0,  max=6.0,  step=0.1,  description='Batch size (kg)',    style=_DW, **S),
    'humidity_g_kg': widgets.FloatSlider(0.0,  min=0.0,  max=0.5,  step=0.05, description='Humidity (g/kg)',    style=_DW, **S),
    'dmc_pct':       widgets.FloatSlider(1.5,  min=1.0,  max=2.0,  step=0.1,  description='DMC (% m/m)',        style=_DW, **S),
    'coating_level': widgets.FloatSlider(0.5,  min=-1.0, max=1.0,  step=0.05, description='Coating level',      style=_DW, **S),
    # Pre-heating
    'ph_T_C':        widgets.FloatSlider(50.0, min=40.0, max=60.0, step=2.0,  description='[PH] T inlet (C)',   style=_DW, **S),
    'ph_flow':       widgets.FloatSlider(250,  min=200,  max=400,  step=10,   description='[PH] Flow (m3/h)',   style=_DW, **S),
    'ph_dur_min':    widgets.FloatSlider(20.0, min=10.0, max=30.0, step=1.0,  description='[PH] Dur (min)',     style=_DW, **S),
    # Spraying
    'sp_T_C':        widgets.FloatSlider(40.0, min=30.0, max=60.0, step=2.0,  description='[SP] T inlet (C)',   style=_DW, **S),
    'sp_flow':       widgets.FloatSlider(250,  min=200,  max=400,  step=10,   description='[SP] Flow (m3/h)',   style=_DW, **S),
    'sp_rate_g_min': widgets.FloatSlider(120,  min=95,   max=150,  step=5,    description='[SP] Spray (g/min)', style=_DW, **S),
    'r_spr_1e6':     widgets.FloatSlider(6.7,  min=0.0,  max=40.0, step=0.5,  description='[SP] r_spray (x1e-6 kg/s)', style=_DW, **S2),
    # Drying
    'dr_T_C':        widgets.FloatSlider(40.0, min=30.0, max=60.0, step=2.0,  description='[DR] T inlet (C)',   style=_DW, **S),
    'dr_flow':       widgets.FloatSlider(250,  min=200,  max=400,  step=10,   description='[DR] Flow (m3/h)',   style=_DW, **S),
    'dr_dur_min':    widgets.FloatSlider(9.0,  min=3.0,  max=15.0, step=1.0,  description='[DR] Dur (min)',     style=_DW, **S),
    'r_dry_1e3':     widgets.FloatSlider(3.1,  min=0.0,  max=10.0, step=0.1,  description='[DR] r_dry (x1e-3 kg/s)',  style=_DW, **S2),
}

_w_t = {'t_proc_min': widgets.FloatSlider(
    30.0, min=0.0, max=90.0, step=0.5, description='Sample time (min)',
    style=_DW, **S, layout=widgets.Layout(width='500px'))}

# ── Panel layout ──────────────────────────────────────────────────────────────
def _lbl(text, color):
    return widgets.HTML(
        f'<b style="background:{color};color:white;padding:3px 10px;'
        f'border-radius:4px;display:inline-block;margin:4px 0">{text}</b>')

_left_col = widgets.VBox([
    _lbl('Particle / Batch', '#4472C4'),
    _w['d_mm'], _w['ssa_cm2g'], _w['T0_C'],
    _w['batch_kg'], _w['humidity_g_kg'], _w['dmc_pct'], _w['coating_level'],
])
_right_col_1 = widgets.VBox([
    _lbl('Pre-heating', '#5B2C6F'),
    _w['ph_T_C'], _w['ph_flow'], _w['ph_dur_min'],
])
_right_col_2 = widgets.VBox([
    _lbl('Spraying  — r_spray adjustable', '#ED7D31'),
    _w['sp_T_C'], _w['sp_flow'], _w['sp_rate_g_min'], _w['r_spr_1e6'],
])
_right_col_3 = widgets.VBox([
    _lbl('Drying  — r_dry adjustable', '#1E8449'),
    _w['dr_T_C'], _w['dr_flow'], _w['dr_dur_min'], _w['r_dry_1e3'],
])
_panel = widgets.HBox([_left_col, _right_col_1, _right_col_2, _right_col_3],
                      layout=widgets.Layout(gap='30px'))

# ── Stage helpers ─────────────────────────────────────────────────────────────
_SC = {'PH': 'royalblue', 'SP': 'darkorange', 'DR': 'seagreen'}

def _stage_at(t, ph_end, sp_end):
    if t <= ph_end: return 'PH', _SC['PH']
    if t <= sp_end: return 'SP', _SC['SP']
    return 'DR', _SC['DR']

def _shade(ax, ph_end, sp_end, t_end):
    ax.axvspan(0, ph_end, alpha=0.07, color='royalblue')
    ax.axvspan(ph_end, sp_end, alpha=0.07, color='darkorange')
    ax.axvspan(sp_end, t_end, alpha=0.07, color='seagreen')
    ax.axvline(ph_end, color='grey', lw=0.7, ls='--', alpha=0.5)
    ax.axvline(sp_end, color='grey', lw=0.7, ls='--', alpha=0.5)
    ax.set_xlabel('Time (min)')

def _stxt(ax, ph_end, sp_end, t_end):
    lo, hi = ax.get_ylim(); yp = lo + (hi - lo) * 0.97
    for pos, lbl, c in [((0+ph_end)/2, 'PH', 'royalblue'),
                         ((ph_end+sp_end)/2, 'SP', 'darkorange'),
                         ((sp_end+t_end)/2, 'DR', 'seagreen')]:
        ax.text(pos, yp, lbl, ha='center', va='top', fontsize=8, color=c, fontweight='bold')

# ── Process simulation ────────────────────────────────────────────────────────
def _run05(d_mm, ssa_cm2g, T0_C, batch_kg, humidity_g_kg, dmc_pct, coating_level,
           ph_T_C, ph_flow, ph_dur_min, sp_T_C, sp_flow, sp_rate_g_min, r_spr_1e6,
           dr_T_C, dr_flow, dr_dur_min, r_dry_1e3):

    dmc_frac    = dmc_pct / 100.0
    humidity    = humidity_g_kg / 1000.0
    qty_sol_kg  = (0.0017 * coating_level + 0.0064) * batch_kg / dmc_frac
    sp_rate_kgs = sp_rate_g_min / 60_000.0
    sp_dur_s    = qty_sol_kg / sp_rate_kgs

    ph_m = ph_flow / 3600.0 * _RHO_AIR
    sp_m = sp_flow / 3600.0 * _RHO_AIR
    dr_m = dr_flow / 3600.0 * _RHO_AIR
    ph_K, sp_K, dr_K = ph_T_C + 273.15, sp_T_C + 273.15, dr_T_C + 273.15

    _PHYS = dict(diameter_eq=d_mm*1e-3, ssa_cm2_g=ssa_cm2g,
                 particle_density=_RHO_PART, cp_particle=_CP_PART,
                 diameter_bed=_D_BED, rho_air=_RHO_AIR, cp_air=_CP_AIR,
                 batch_size=batch_kg)

    p_ph = ProcessParameters(**_PHYS, air_flow_rates=(ph_m,)*3,
                             air_temperatures=(ph_K,)*3, air_inlet_moisture=(humidity,)*3)
    res_ph = run_preheating(p_ph, duration=ph_dur_min*60.0, T_particle_init=T0_C+273.15)

    p_sp = ProcessParameters(**_PHYS, air_flow_rates=(sp_m,)*3,
                             air_temperatures=(sp_K,)*3, air_inlet_moisture=(humidity,)*3,
                             spray_rate=sp_rate_kgs, dry_matter_conc=dmc_frac,
                             r_spraying=r_spr_1e6*1e-6)
    res_sp = run_spraying(p_sp, duration=sp_dur_s, T_particle_init=res_ph.T_particle[-1])

    p_dr = ProcessParameters(**_PHYS, air_flow_rates=(dr_m,)*3,
                             air_temperatures=(dr_K,)*3, air_inlet_moisture=(humidity,)*3,
                             r_drying=r_dry_1e3*1e-3)
    res_dr = run_drying(p_dr, duration=dr_dur_min*60.0,
                        Y_particle_init=res_sp.Y_particle[-1],
                        Y_gas_init=res_sp.Y_gas[-1],
                        M_coating_init=res_sp.M_coating[-1],
                        T_particle_init=res_sp.T_particle[-1])

    t_ph  = res_ph.t / 60.0
    t_sp  = (res_sp.t + res_ph.t[-1]) / 60.0
    t_dr  = (res_dr.t + res_ph.t[-1] + res_sp.t[-1]) / 60.0
    t_all = np.concatenate([t_ph, t_sp, t_dr])
    ph_end, sp_end, t_end = t_ph[-1], t_sp[-1], t_dr[-1]

    # No-loss WG reference: linear ramp during spray, flat plateau during dry
    WG_max_noloss = qty_sol_kg * dmc_frac / batch_kg * 100.0
    WG_noloss = np.concatenate([
        np.zeros_like(t_ph),
        np.linspace(0.0, WG_max_noloss, len(t_sp)),
        np.full(len(t_dr), WG_max_noloss),
    ])

    if _w_t['t_proc_min'].value > t_end:
        _w_t['t_proc_min'].value = t_end
    _w_t['t_proc_min'].max = t_end

    t_step = [0, ph_end, ph_end, sp_end, sp_end, t_end]
    T_step = [ph_T_C, ph_T_C, sp_T_C, sp_T_C, dr_T_C, dr_T_C]
    T_prod = np.concatenate([res_ph.T_particle-273.15, res_sp.T_particle-273.15, res_dr.T_particle-273.15])
    T_gas  = np.concatenate([res_ph.T_gas-273.15, res_sp.T_gas-273.15, res_dr.T_gas-273.15])
    Y_part = np.concatenate([np.zeros_like(t_ph), res_sp.Y_particle*100, res_dr.Y_particle*100])
    Y_gas_ = np.concatenate([np.zeros_like(t_ph), res_sp.Y_gas*100, res_dr.Y_gas*100])
    WG     = np.concatenate([np.zeros_like(t_ph), res_sp.M_coating/batch_kg*100,
                                                   res_dr.M_coating/batch_kg*100])

    _sim_cache.update(dict(t_all=t_all, WG=WG, WG_noloss=WG_noloss,
                           ph_end=ph_end, sp_end=sp_end, t_end=t_end,
                           ssa_cm2g=ssa_cm2g, batch_kg=batch_kg))

    fig, axes = plt.subplots(2, 2, figsize=(12, 5))
    fig.suptitle(
        f'Fluid Bed Coating — Demo  |  '
        f'r_spray = {r_spr_1e6:.1f} x1e-6 kg/s  |  r_dry = {r_dry_1e3:.2f} x1e-3 kg/s',
        fontsize=12, fontweight='bold')

    ax = axes[0, 0]
    _shade(ax, ph_end, sp_end, t_end)
    ax.step(t_step, T_step, color='tomato', lw=1.5, ls='--', where='post', label='T inlet')
    ax.plot(t_all, T_gas, color='tomato', lw=1.5, alpha=0.5, label='T outlet')
    ax.plot(t_all, T_prod, color='steelblue', lw=2.5, label='T product')
    ax.set_ylabel('Temperature (C)'); ax.set_title('Temperature profiles')
    ax.legend(fontsize=8, loc='lower left'); ax.grid(True, alpha=0.3); _stxt(ax, ph_end, sp_end, t_end)

    ax = axes[0, 1]
    _shade(ax, ph_end, sp_end, t_end)
    ax.plot(t_all, Y_part, color='darkorange', lw=2.5)
    ax.set_ylabel('Acetone on particles (wt %)'); ax.set_title('Particle acetone')
    ax.grid(True, alpha=0.3); _stxt(ax, ph_end, sp_end, t_end)

    ax = axes[1, 0]
    _shade(ax, ph_end, sp_end, t_end)
    ax.plot(t_all, Y_gas_, color='cadetblue', lw=2.5)
    ax.set_ylabel('Acetone in gas (wt %)'); ax.set_title('Gas-phase acetone')
    ax.grid(True, alpha=0.3); _stxt(ax, ph_end, sp_end, t_end)

    ax = axes[1, 1]
    _shade(ax, ph_end, sp_end, t_end)
    ax.plot(t_all, WG_noloss, color='mediumpurple', lw=1.5, ls=':', alpha=0.8, label='WG no loss')
    ax.plot(t_all, WG, color='mediumpurple', lw=2.5, label='WG model')
    ax.set_ylabel('Coating weight gain (%)'); ax.set_title('Coating WG')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3); _stxt(ax, ph_end, sp_end, t_end)

    plt.tight_layout(); plt.show(); plt.close()

    wg_sp  = res_sp.M_coating[-1] / batch_kg * 100
    wg_fin = res_dr.M_coating[-1] / batch_kg * 100
    print(f'Spray: {qty_sol_kg*1000:.0f} g in {sp_dur_s/60:.1f} min  |  Total: {t_end:.1f} min')
    print(f'r_spray = {r_spr_1e6:.1f} x1e-6 kg/s  |  r_dry = {r_dry_1e3:.2f} x1e-3 kg/s')
    print(f'WG end-spray: {wg_sp:.3f}%  |  WG final: {wg_fin:.3f}%  |  WG no-loss: {WG_max_noloss:.3f}%')
    print('Move the Sample time slider below to update the dissolution profile.')


# ── Dissolution panel ─────────────────────────────────────────────────────────
def _run_disso(t_proc_min):
    if not _sim_cache:
        print('Move any process slider to run the simulation first.')
        return

    t_all     = _sim_cache['t_all']
    WG        = _sim_cache['WG']
    WG_noloss = _sim_cache['WG_noloss']
    ph_end    = _sim_cache['ph_end']
    sp_end    = _sim_cache['sp_end']
    t_end     = _sim_cache['t_end']
    ssa       = _sim_cache['ssa_cm2g']

    t_s        = float(np.clip(t_proc_min, 0.0, t_end))
    wg_at_t    = float(np.interp(t_s, t_all, WG))
    wg_nl_at_t = float(np.interp(t_s, t_all, WG_noloss))
    stage_name, stage_color = _stage_at(t_s, ph_end, sp_end)

    t_diss, F_diss, k    = _dissolution_curve(wg_at_t / 100.0, ssa)
    _,      F_noloss, _  = _dissolution_curve(wg_nl_at_t / 100.0, ssa)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(
        f'Virtual Sample — t = {t_s:.1f} min  [{stage_name}]  '
        f'WG = {wg_at_t:.3f}%  (no-loss: {wg_nl_at_t:.3f}%)   k = {k:.4e} s-1',
        fontsize=11, fontweight='bold')

    # Left: WG profile + sample marker
    ax = axes[0]
    ax.axvspan(0, ph_end, alpha=0.07, color='royalblue')
    ax.axvspan(ph_end, sp_end, alpha=0.07, color='darkorange')
    ax.axvspan(sp_end, t_end, alpha=0.07, color='seagreen')
    ax.axvline(ph_end, color='grey', lw=0.7, ls='--', alpha=0.5)
    ax.axvline(sp_end, color='grey', lw=0.7, ls='--', alpha=0.5)
    ax.plot(t_all, WG_noloss, color='mediumpurple', lw=1.5, ls=':', alpha=0.8, label='WG no loss')
    ax.plot(t_all, WG, color='mediumpurple', lw=2.0, label='WG model')
    bw = max(t_end * 0.008, 0.3)
    ax.axvspan(t_s - bw, t_s + bw, alpha=0.55, color=stage_color, label=f'Sample [{stage_name}]')
    ax.axvline(t_s, color=stage_color, lw=1.5)
    ax.axhline(wg_at_t, color=stage_color, lw=1.0, ls=':', alpha=0.6)
    ax.annotate(f'{wg_at_t:.3f}%', xy=(t_s, wg_at_t),
                xytext=(t_s + t_end*0.04, wg_at_t), fontsize=9, color=stage_color, va='center')
    lo, hi = ax.get_ylim(); yp = lo + (hi - lo) * 0.97
    for pos, lbl, c in [((0+ph_end)/2, 'PH', 'royalblue'),
                         ((ph_end+sp_end)/2, 'SP', 'darkorange'),
                         ((sp_end+t_end)/2, 'DR', 'seagreen')]:
        ax.text(pos, yp, lbl, ha='center', va='top', fontsize=8, color=c, fontweight='bold')
    ax.set_xlabel('Process time (min)'); ax.set_ylabel('Coating WG (%)')
    ax.set_title('Coating evolution — sample point')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    # Right: dissolution curve (model vs no-loss)
    ax = axes[1]
    try:
        from matplotlib.colors import to_rgba
        ax.set_facecolor(to_rgba(stage_color, 0.06))
    except Exception:
        pass
    ax.plot(t_diss, F_noloss, color='dimgrey', lw=1.5, ls=':', alpha=0.8, label='No loss')
    ax.plot(t_diss, F_diss, color=stage_color, lw=2.5, label='Model')
    for tgt, ls in [(50, '--'), (80, ':')]:
        if F_diss[-1] >= tgt:
            t_m = float(np.interp(tgt, F_diss, t_diss))
            ax.axhline(tgt, color='grey', lw=0.8, ls=ls, alpha=0.5)
            ax.axvline(t_m, color='grey', lw=0.8, ls=ls, alpha=0.5)
            ax.text(t_m + 3, tgt + 1.5, f't{tgt} = {t_m:.0f} min', fontsize=8, color='dimgrey')
    ax.set_xlim(0, _DISS['Total_min']); ax.set_ylim(0, 105)
    ax.set_xlabel('Dissolution time (min)'); ax.set_ylabel('Drug released (%)')
    ax.set_title(f'First-order dissolution  [{stage_name}]')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show(); plt.close()

    t50 = float(np.interp(50, F_diss, t_diss)) if F_diss[-1] >= 50 else float('nan')
    t80 = float(np.interp(80, F_diss, t_diss)) if F_diss[-1] >= 80 else float('nan')
    t50_s = f'{t50:.0f} min' if not np.isnan(t50) else '> 240 min'
    t80_s = f'{t80:.0f} min' if not np.isnan(t80) else '> 240 min'
    print(f't50 = {t50_s}  |  t80 = {t80_s}  |  k = {k:.4e} s-1  |  stage = {stage_name}')


# ── Wire up and display ───────────────────────────────────────────────────────
_out_proc  = widgets.interactive_output(_run05, _w)
_out_disso = widgets.interactive_output(_run_disso, _w_t)

_disso_header = widgets.VBox([
    widgets.HTML(
        '<b style="background:#8E44AD;color:white;padding:3px 10px;'
        'border-radius:4px;display:inline-block;margin:8px 0">'
        'Dissolution — Virtual Sample Withdrawal</b>'
        '<br><small>Slide to any process time. '
        'Slider max auto-adjusts to the actual total process duration.</small>'
    ),
    _w_t['t_proc_min'],
])

clear_output(wait=True)
display(_panel, _out_proc, _disso_header, _out_disso)